In [43]:
import numpy as np
import polars as pl
import matplotlib.pyplot as plt
from datetime import datetime, timedelta
import yfinance as yf
import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader
import torch.optim as optim
import plotly.graph_objects as go


In [44]:
def get_data(ticker) -> pl.DataFrame:
    ticker_data = yf.Ticker(ticker)
    expirations = ticker_data.options
    
    history = ticker_data.history(period="1d")

    if history.empty:
        raise ValueError(f"No price data found for {ticker}")
    spot_price = history["Close"].iloc[-1]

    all_data = []
    today = datetime.now().date()

    for exp in expirations:
        chain = ticker_data.option_chain(exp)
        
        calls = pl.from_pandas(chain.calls).with_columns(pl.lit("call").alias("type"))
        puts = pl.from_pandas(chain.puts).with_columns(pl.lit("put").alias("type"))
        
        calls = calls.with_columns(pl.lit(exp).alias("expiration_date"))
        puts = puts.with_columns(pl.lit(exp).alias("expiration_date"))
        
        combined = pl.concat([calls, puts], how="vertical_relaxed")
        combined = combined.with_columns(pl.lit(exp).alias("expiration_date"))
        all_data.append(combined)

    df = pl.concat(all_data, how="vertical_relaxed")

    return df.with_columns(
        pl.lit(spot_price).alias("spot_price"),
        pl.col("expiration_date").str.to_date("%Y-%m-%d"),
        pl.lit(today).alias("current_date")
    ).with_columns(
        days_to_expiry=(pl.col("expiration_date") - pl.col("current_date")).dt.total_days()
    ).with_columns(
        time_to_expiry=pl.col("days_to_expiry") / 365.0
    ).select([
        "contractSymbol", "strike", "bid", "ask", "volume", 
        "type", "days_to_expiry", "time_to_expiry", 
        "spot_price", "impliedVolatility"
    ])

In [45]:
raw_options_df = get_data("SPY")

# Estimeates for Dividend Yield and Risk-Free Rate
r_estimate = 0.045  
q_estimate = 0.013  # Div yield

clean_df = (
    raw_options_df.lazy()
    .filter(
        (pl.col("volume") > 0) & 
        (pl.col("bid") > 0) &
        (pl.col("days_to_expiry") > 0) 
    )
    .with_columns(
        mid_price=(pl.col("bid") + pl.col("ask")) / 2.0,
        tau=pl.col("days_to_expiry") / 365.25,
        r=pl.lit(r_estimate),
        q=pl.lit(q_estimate)
    )
    .with_columns(
        forward_price=pl.col("spot_price") * ((pl.col("r") - pl.col("q")) * pl.col("tau")).exp()
    )
    .with_columns(
        log_moneyness=(pl.col("strike") / pl.col("forward_price")).log()
    )
    .with_columns(
        market_variance = pl.col("impliedVolatility")**2 * pl.col("tau")
    )
    .collect()
)

print(clean_df.head())

shape: (5, 17)
┌──────────────┬────────┬────────┬────────┬───┬───────┬──────────────┬──────────────┬──────────────┐
│ contractSymb ┆ strike ┆ bid    ┆ ask    ┆ … ┆ q     ┆ forward_pric ┆ log_moneynes ┆ market_varia │
│ ol           ┆ ---    ┆ ---    ┆ ---    ┆   ┆ ---   ┆ e            ┆ s            ┆ nce          │
│ ---          ┆ f64    ┆ f64    ┆ f64    ┆   ┆ f64   ┆ ---          ┆ ---          ┆ ---          │
│ str          ┆        ┆        ┆        ┆   ┆       ┆ f64          ┆ f64          ┆ f64          │
╞══════════════╪════════╪════════╪════════╪═══╪═══════╪══════════════╪══════════════╪══════════════╡
│ SPY260414C00 ┆ 580.0  ┆ 104.76 ┆ 107.58 ┆ … ┆ 0.013 ┆ 686.160088   ┆ -0.168083    ┆ 0.004332     │
│ 580000       ┆        ┆        ┆        ┆   ┆       ┆              ┆              ┆              │
│ SPY260414C00 ┆ 605.0  ┆ 79.76  ┆ 82.58  ┆ … ┆ 0.013 ┆ 686.160088   ┆ -0.125883    ┆ 0.002569     │
│ 605000       ┆        ┆        ┆        ┆   ┆       ┆              ┆      

In [46]:

class IVSurfaceNet(nn.Module):
    def __init__(self, hidden_size=64):
        super(IVSurfaceNet, self).__init__()
        
     
        self.layer1 = nn.Linear(2, hidden_size)
        self.layer2 = nn.Linear(hidden_size, hidden_size)
        self.layer3 = nn.Linear(hidden_size, hidden_size)
        

        self.output_layer = nn.Linear(hidden_size, 1)
        
        self.activation = nn.Softplus()

    def forward(self, x):
        x = self.activation(self.layer1(x))
        x = self.activation(self.layer2(x))
        x = self.activation(self.layer3(x))
        
        
        w = self.activation(self.output_layer(x))
        
        return w

In [47]:

class ArbitrageFreeLoss(nn.Module):
    def __init__(self, lambda_calendar=1.0, lambda_butterfly=1.0):
        super(ArbitrageFreeLoss, self).__init__()
        self.mse = nn.MSELoss()
        self.lambda_calendar = lambda_calendar
        self.lambda_butterfly = lambda_butterfly

    def forward(self, model, inputs, w_market):
        """
        inputs: Tensor of shape (N, 2) where col 0 is k, col 1 is tau.
                MUST have requires_grad=True before passing to the model.
        w_market: Tensor of shape (N, 1) containing target market variance.
        """
        w_pred = model(inputs)
        
        gradients = torch.autograd.grad(
            outputs=w_pred, 
            inputs=inputs,
            grad_outputs=torch.ones_like(w_pred),
            create_graph=True, 
        )[0]
        
        dw_dk = gradients[:, 0:1]
        dw_dtau = gradients[:, 1:2]
        d2w_dk2 = torch.autograd.grad(
            outputs=dw_dk,
            inputs=inputs,
            grad_outputs=torch.ones_like(dw_dk),
            create_graph=True,
            retain_graph=True
        )[0][:, 0:1]
        
        k = inputs[:, 0:1]

        calendar_penalty = torch.relu(-dw_dtau) # calendar spread constraint: dw/dtau >= 0
        
        # Butterfly spread constraint
        eps = 1e-6
        w_safe = w_pred + eps 
        
        term1 = (1.0 - (k / (2.0 * w_safe)) * dw_dk) ** 2
        term2 = 0.25 * (1.0 / w_safe + 0.25) * (dw_dk ** 2)
        term3 = 0.5 * d2w_dk2
        
        g_k = term1 - term2 + term3
        butterfly_penalty = torch.relu(-g_k)
        
        fit_loss = self.mse(w_pred, w_market)
        cal_loss = self.lambda_calendar * torch.mean(calendar_penalty)
        fly_loss = self.lambda_butterfly * torch.mean(butterfly_penalty)
        
        total_loss = fit_loss + cal_loss + fly_loss
        
        return total_loss, fit_loss, cal_loss, fly_loss

In [48]:
inputs_tensor = torch.tensor(
    clean_df.select(["log_moneyness", "tau"]).to_numpy(), 
    dtype=torch.float32
)

w_market_tensor = torch.tensor(
    clean_df.select("market_variance").to_numpy(), 
    dtype=torch.float32
)

dataset = TensorDataset(inputs_tensor, w_market_tensor)
dataloader = DataLoader(dataset, batch_size=256, shuffle=True)

model = IVSurfaceNet(hidden_size=64)
criterion = ArbitrageFreeLoss(lambda_calendar=1.0, lambda_butterfly=1.0)
optimiser = optim.Adam(model.parameters(), lr=1e-3)

epochs = 100


for epoch in range(epochs):
    model.train()
    epoch_total_loss = 0.0
    epoch_fit_loss = 0.0
    
    for batch_inputs, batch_w_market in dataloader:
        optimiser.zero_grad()
        
        batch_inputs.requires_grad_(True)
        
        total_loss, fit_loss, cal_loss, fly_loss = criterion(
            model, 
            batch_inputs, 
            batch_w_market
        )
        
        total_loss.backward()
        
        optimiser.step()
        
        epoch_total_loss += total_loss.item()
        epoch_fit_loss += fit_loss.item()
        
    if (epoch + 1) % 10 == 0:
        avg_loss = epoch_total_loss / len(dataloader)
        avg_fit = epoch_fit_loss / len(dataloader)
        print(f"Epoch {epoch+1}/{epochs} | Total Loss: {avg_loss:.6f} | Fit (MSE): {avg_fit:.6f}")

Epoch 10/100 | Total Loss: 0.008540 | Fit (MSE): 0.008540
Epoch 20/100 | Total Loss: 0.002610 | Fit (MSE): 0.002610
Epoch 30/100 | Total Loss: 0.002521 | Fit (MSE): 0.002521
Epoch 40/100 | Total Loss: 0.002535 | Fit (MSE): 0.002535
Epoch 50/100 | Total Loss: 0.002422 | Fit (MSE): 0.002422
Epoch 60/100 | Total Loss: 0.002382 | Fit (MSE): 0.002382
Epoch 70/100 | Total Loss: 0.002429 | Fit (MSE): 0.002429
Epoch 80/100 | Total Loss: 0.002413 | Fit (MSE): 0.002413
Epoch 90/100 | Total Loss: 0.002380 | Fit (MSE): 0.002380
Epoch 100/100 | Total Loss: 0.002423 | Fit (MSE): 0.002423


In [49]:
from scipy.optimize import minimize

market_k = clean_df["log_moneyness"].to_numpy()
market_tau = clean_df["tau"].to_numpy()
market_iv = clean_df["impliedVolatility"].to_numpy()

def ssvi_objective(params):
    sigma_atm, rho, eta = params
    
    theta = (sigma_atm ** 2) * market_tau
    phi = eta / np.sqrt(theta + 1e-6)
    
    inner_sqrt = np.sqrt((phi * market_k + rho)**2 + (1.0 - rho**2))
    w = (theta / 2.0) * (1.0 + rho * phi * market_k + inner_sqrt)
    
    iv_predicted = np.sqrt(w / market_tau)
    
    mse = np.mean((market_iv - iv_predicted) ** 2)
    return mse

initial_guess = [0.20, -0.5, 0.5]
bounds = ((0.01, 2.0), (-0.99, 0.99), (0.01, 5.0))

print("Calibrating SSVI to market data...")
result = minimize(
    ssvi_objective, 
    initial_guess, 
    bounds=bounds, 
    method='L-BFGS-B'
)

calibrated_sigma, calibrated_rho, calibrated_eta = result.x

print(f"Calibration Successful: {result.success}")
print(f"Optimal ATM Vol (\u03C3_atm): {calibrated_sigma:.4f}")
print(f"Optimal Skew (\u03C1): {calibrated_rho:.4f}")
print(f"Optimal Convexity (\u03B7): {calibrated_eta:.4f}")

Calibrating SSVI to market data...
Calibration Successful: True
Optimal ATM Vol (σ_atm): 0.1357
Optimal Skew (ρ): -0.4016
Optimal Convexity (η): 2.7289


In [50]:
def calculate_ssvi_surface(k, tau, sigma_atm, rho, eta):
    
    theta = (sigma_atm ** 2) * tau
    
    phi = eta / np.sqrt(theta + 1e-6) 
    
    inner_sqrt = np.sqrt((phi * k + rho)**2 + (1.0 - rho**2))
    w = (theta / 2.0) * (1.0 + rho * phi * k + inner_sqrt)
    
    iv = np.sqrt(w / tau)
    return iv

ssvi_iv_pred = calculate_ssvi_surface(K_grid, Tau_grid, calibrated_sigma, calibrated_rho, calibrated_eta)

k_vals = np.linspace(-0.5, 0.5, 50)
tau_vals = np.linspace(0.05, 2.0, 50)
K_grid, Tau_grid = np.meshgrid(k_vals, tau_vals)

grid_inputs = np.column_stack((K_grid.flatten(), Tau_grid.flatten()))
tensor_inputs = torch.tensor(grid_inputs, dtype=torch.float32)

model.eval()
with torch.no_grad():
    w_pred = model(tensor_inputs).numpy().flatten()

tau_flattened = Tau_grid.flatten()
iv_pred = np.sqrt(w_pred / (tau_flattened + 1e-6)) 

IV_surface = iv_pred.reshape(K_grid.shape)

# Plot
fig = go.Figure()

# NN surface
fig.add_trace(go.Surface(
    z=IV_surface, 
    x=K_grid, 
    y=Tau_grid, 
    colorscale='Plasma',
    name='Neural Network',
    showscale=False,
    opacity=0.9 
))

# SSVI
fig.add_trace(go.Surface(
    z=ssvi_iv_pred, 
    x=K_grid, 
    y=Tau_grid, 
    colorscale='Viridis', 
    name='SSVI Benchmark',
    showscale=False,
    opacity=0.5 
))

fig.add_trace(go.Scatter3d(
    x=market_k,
    y=market_tau,
    z=market_iv,
    mode='markers',
    marker=dict(
        size=2,
        color='red',         
        opacity=0.9,
        line=dict(width=0.5, color='darkred')
    ),
    name='Market Data (True IV)'
))

# Format the layout
fig.update_layout(
    title='Market Data vs. Interpolation Models',
    autosize=False,
    width=900,
    height=800,
    scene=dict(
        xaxis_title='Log-Moneyness (k)',
        yaxis_title='Time to Maturity (\u03C4)',
        zaxis_title='Implied Volatility (\u03C3)'
    )
)

fig.show()